<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 09 · Financial Time Series

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the chapter examples in a Colab-ready format so that you
can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time so the time-series objects are
  created in order.
- Add your own cells for experiments or refactorings.
- Use the book text for the broader discussion of data handling choices.


This chapter shows how to turn raw end-of-day prices into well-structured time
series for analysis.


# Why Clean Time Series Matter


Reliable analysis starts with proper datetime indexes, expected shapes, and
explicit missing values.


# Inspecting the Raw EOD Data


The CSV contains a date column followed by price levels for several
liquid instruments.


# Loading Financial Time Series with a Local/Remote Fallback


Prefer the local CSV first and fall back to the remote copy only when the file
is missing.


In [ ]:
from pathlib import Path

In [ ]:
import pandas as pd

In [ ]:
# Path to the local CSV file shipped with this project.
LOCAL_EOD = Path("..") / "data" / "eod_data.csv"


In [ ]:
# Remote fallback URL that exposes the same CSV structure.
REMOTE_EOD = "https://hilpisch.com/eod_data.csv"

In [ ]:
# Decide at runtime whether to load from the local file or from the remote
# URL.
if LOCAL_EOD.exists():
    eod_source = LOCAL_EOD
else:
    eod_source = REMOTE_EOD

In [ ]:
eod_source

In [ ]:
# Load the CSV without any time-series-specific arguments.
df_raw = pd.read_csv(eod_source)


In [ ]:
df_raw.head()

In [ ]:
df_raw.dtypes

In [ ]:
# The index is a simple `RangeIndex` (0, 1, 2, ...) and the `Date` column is a
# plain object column, not a datetime index.
df_raw.index

# First Attempt: Reading Without Index or Date Parsing


A raw import keeps the date column as plain text, which breaks date-based
indexing.


In [ ]:
# With a `RangeIndex`, a string label like `"2016-01-04"` is not a valid index
# key, so `.loc[]` raises a `KeyError`.
try:
    df_raw.loc["2016-01-04"]
except KeyError as exc:
    print(f"Caught expected error: {exc}")

# Fixing Date Parsing and the Index at Import Time


Read the file again with `parse_dates` and `index_col` so the index becomes a
`DatetimeIndex`.


In [ ]:
df = pd.read_csv(
    eod_source,
    parse_dates=["Date"],
    index_col="Date",
# Parse the `Date` column as datetimes and use it as the index directly when
# reading the CSV.
)

In [ ]:
df.head()

In [ ]:
# The resulting index is a `DatetimeIndex` named `"Date"`, which enables time-
# aware operations.
df.index

In [ ]:
df.dtypes.iloc[:4]

# What If You Forget `parse_dates` or `index_col`?


If you imported the CSV with defaults, convert the date column afterward and
set it as the index.


In [ ]:
# Selecting a single date returns a `Series` of prices for that day.
df.loc["2016-01-04"]


In [ ]:
# Partial-string indexing with `"2016-01"` selects all trading days in January
# 2016.
df.loc["2016-01"]

# Inspecting Time-Series Structure Systematically


Check shape, columns, index bounds, and metadata before you move on to
analysis.


In [ ]:
# Start from a naïve import that treats `Date` as a string column.
df_fix = pd.read_csv(eod_source)


In [ ]:
df_fix["Date"].head()

In [ ]:
# Convert the `Date` column to datetime objects explicitly.
df_fix["Date"] = pd.to_datetime(df_fix["Date"])

In [ ]:
# Move the converted column into the index and sort by date to restore
# canonical order.
df_fix = df_fix.set_index("Date").sort_index()

In [ ]:
# The resulting index now behaves like the one you would have obtained with
# `parse_dates` and `index_col` at import time.
df_fix.index

In [ ]:
# The number of rows and columns tells you how many trading days and
# instruments the dataset covers.
df.shape

In [ ]:
df.columns  # Column labels correspond to asset tickers and currency pairs.

In [ ]:
# The minimum and maximum index values reveal the time span.
df.index.min(), df.index.max()


In [ ]:
# `.info()` summarizes dtypes and memory usage, which can hint at problematic
# columns (for example, unexpected `object` dtypes).
df.info()

# Creating a Sub-Sample with Missing Data


A small sample with injected `NaN` values makes it easier to test cleaning
logic.


In [ ]:
# Create a business-day date range from the first to the last observation.
all_bdays = pd.date_range(df.index.min(), df.index.max(), freq="B")

In [ ]:
# Reindex the data onto this calendar, introducing `NaN` values where no
# observation is available.
aligned = df.reindex(all_bdays)

In [ ]:
# Count how many business days have at least one missing value; unexpected
# gaps can indicate data issues that deserve closer inspection.
aligned.isna().any(axis=1).sum()

In [ ]:
import numpy as np

In [ ]:
# Start from a two-column subset that is easy to inspect.
prices_sub = df[["AAPL", "SPY"]].copy()


In [ ]:
rng = np.random.default_rng(seed=42)

In [ ]:
# Create a Boolean mask that flags roughly 5% of dates at random.
mask = rng.random(len(prices_sub)) < 0.05

In [ ]:
# Introduce missing values in the `AAPL` column for those dates.
prices_sub.loc[mask, "AAPL"] = np.nan


In [ ]:
prices_sub.isna().sum()

# Basic Missing-Data Strategies


Drop, forward-fill, and interpolate each serve a different purpose.


In [ ]:
# Dropping rows removes any date where `AAPL` is missing; this is conservative
# but may shorten your series.
dropped = prices_sub.dropna()

In [ ]:
dropped.head()

In [ ]:
# Forward-filling propagates the last known `AAPL` price; this is often
# appropriate for carry-like quantities or holdings that change infrequently.
ffilled = prices_sub.ffill()

In [ ]:
ffilled.loc[mask, "AAPL"].head()

In [ ]:
# Time-based interpolation guesses intermediate values between known
# observations; this can be useful for charting but should be applied with
# care in risk or PnL calculations.
interpolated = prices_sub.interpolate(method="time")

# Returns and Log Returns from EOD Prices


Returns and log returns are the basic building blocks for most later analyses.


In [ ]:
# Work with a subset of assets to keep the output compact.
prices = df[["AAPL", "SPY"]].copy()


In [ ]:
# `.pct_change()` computes period-over-period simple returns for each column,
# aligned to the original index.
rets = prices.pct_change()

In [ ]:
rets.head()

In [ ]:
# `.describe().T` summarizes basic statistics per asset; focusing on `mean`
# and `std` already gives you a sense of drift and volatility.
rets.describe().T[["mean", "std"]]

In [ ]:
import numpy as np

In [ ]:
# Take the element-wise logarithm of the price ratio `P_t / P_{t-1}` to obtain
# log returns; for small moves they are close to simple returns but add nicely
# over time.
log_rets = np.log(prices / prices.shift(1))

In [ ]:
log_rets.head()

In [ ]:
# Select a common start date and record the corresponding prices for a subset
# of assets.
base = df.loc["2016-01-04", ["AAPL", "SPY", "GLD", "TLT"]]

In [ ]:
# Divide each column by its starting value so that all series begin at 1.0;
# plotting `norm` shows relative growth paths on the same scale.
norm = df[["AAPL", "SPY", "GLD", "TLT"]].div(base)

In [ ]:
norm.head()

# Resampling and Time Aggregation


Resampling converts a daily series into a coarser frequency when you need it.


In [ ]:
spy_daily = df["SPY"]  # Work with a single price series for clarity.

In [ ]:
# `"M"` resamples to calendar month-end, taking the last available observation
# in each month.
spy_monthly = spy_daily.resample("M").last()

In [ ]:
spy_monthly.head()

In [ ]:
# `"W-FRI"` resamples to weekly data anchored on Fridays; other anchors such
# as `"W-MON"` are also possible.
spy_weekly = spy_daily.resample("W-FRI").last()

In [ ]:
spy_monthly_rets = spy_monthly.pct_change()

In [ ]:
spy_monthly_rets.describe()[["mean", "std"]]

# Rolling Statistics and Simple Moving Averages


Rolling windows smooth noise and make trend changes easier to inspect.


In [ ]:
# Choose two windows that roughly correspond to one and three trading months.
window_short = 21

In [ ]:
window_long = 63

In [ ]:
# Compute a short-term simple moving average (SMA) of the SPY price.
spy_sma_short = spy_daily.rolling(window_short).mean()

In [ ]:
# Compute a longer-term SMA for comparison.
spy_sma_long = spy_daily.rolling(window_long).mean()


In [ ]:
# Estimate annualized rolling volatility from daily returns over a 63-day
# window (assuming 252 trading days per year).
spy_roll_vol = spy_daily.pct_change().rolling(63).std() * (252**0.5)

In [ ]:
sma_df = pd.DataFrame(
    {"price": spy_daily, "sma_short": spy_sma_short, "sma_long": spy_sma_long}
# Collect the original price and both SMAs in one `DataFrame` and drop the
# initial rows where SMAs are undefined.
).dropna()

In [ ]:
# Define a simple indicator: `1` when the short SMA is above the long SMA, `0`
# otherwise. Later chapters use similar constructions to derive trading
# positions.
sma_df["signal"] = (sma_df["sma_short"] > sma_df["sma_long"]).astype(int)

In [ ]:
sma_df["signal"].value_counts()

# Visualizing Time Series and Indicators with pandas


Plot prices, moving averages, and strategy equity curves to see the signal
directly.


## Price Overview


A simple price plot gives you a fast view of the main series.


In [ ]:
# Choose a subset of columns to keep the figure readable.
price_cols = ["AAPL", "SPY", "GLD", "TLT"]


In [ ]:
# Call `.plot()` on the `DataFrame`; `pandas` creates a figure and axis and
# draws one line per column with a legend.
ax = df[price_cols].plot(figsize=(7, 4))

In [ ]:
# Customize title and axis labels using the underlying `Axes` object.
ax.set_title("Selected End-of-Day Prices")

In [ ]:
ax.set_xlabel("Date")

In [ ]:
ax.set_ylabel("Price level")

## Moving Averages and Equity Curves


Moving averages and a basic equity curve show how a simple rule would have
behaved.


In [ ]:
# Plot the SPY price series first and keep a reference to the returned `Axes`.
ax = spy_daily.plot(figsize=(7, 4), color="tab:gray", label="SPY")

In [ ]:
# Plot the short SMA on the same axes, controlling color and label explicitly.
spy_sma_short.plot(ax=ax, color="tab:blue", label="21-day SMA")

In [ ]:
# Plot the long SMA similarly, resulting in three labeled lines in one figure.
spy_sma_long.plot(ax=ax, color="tab:orange", label="63-day SMA")

In [ ]:
ax.set_title("SPY with Short and Long SMAs")

In [ ]:
ax.set_xlabel("Date")

In [ ]:
ax.set_ylabel("Price level")

In [ ]:
ax.legend(loc="upper left")

In [ ]:
# Compute daily SPY returns and drop the initial `NaN`.
spy_rets = spy_daily.pct_change().dropna()


In [ ]:
# Turn the SMA crossover condition into a `float` position: `1.0` for long,
# `0.0` for flat.
pos = (sma_df["sma_short"] > sma_df["sma_long"]).astype(float)

In [ ]:
pos = (sma_df["sma_short"] > sma_df["sma_long"]).astype(float)
base_rets = spy_rets.reindex(sma_df.index).fillna(0.0)
strat_rets = pos.shift(1).fillna(0.0) * base_rets


In [ ]:
# Compound strategy returns into an equity curve starting at 1.0.
equity_strategy = (1.0 + strat_rets).cumprod()

In [ ]:
equity_buy_hold = (
    1.0 + base_rets.reindex(equity_strategy.index).fillna(0.0)
).cumprod()


In [ ]:
# Collect both equity curves into a `DataFrame` for plotting.
equity = pd.DataFrame(
    {"strategy": equity_strategy, "buy_and_hold": equity_buy_hold}
)

In [ ]:
# Use `.plot()` to draw both curves with a shared legend and axes.
ax = equity.plot(figsize=(7, 4))


In [ ]:
ax.set_title("SMA Strategy vs. Buy-and-Hold (SPY)")

In [ ]:
ax.set_xlabel("Date")

In [ ]:
ax.set_ylabel("Equity (starting at 1.0)")

# Grouping by Calendar Periods


Groupby operations let you summarize returns by month, year, or any other
calendar bucket.


In [ ]:
# Compute daily returns for all assets and drop the initial `NaN` row.
daily_rets = df.pct_change().dropna()

In [ ]:
# Group returns by calendar month using `pd.Grouper(freq="M")` and aggregate
# mean and standard deviation for SPY and TLT.
monthly_summary = daily_rets.groupby(
    pd.Grouper(freq="M")
)[["SPY", "TLT"]].agg(["mean", "std"])

In [ ]:
# Inspect the first few rows to see monthly summary statistics by asset.
monthly_summary.head()


# Correlation Analysis Across Assets


Correlation matrices and rolling correlations reveal how assets move together.


In [ ]:
# Compute daily returns for all columns and drop the initial `NaN` row.
corr_mat = daily_rets.corr()


In [ ]:
# `.corr()` derives the Pearson correlation matrix across assets.
corr_mat.loc[["SPY", "TLT", "GLD"], ["SPY", "TLT", "GLD"]]

In [ ]:
pair = daily_rets[["SPY", "TLT"]]  # Focus on a pair of assets for readability.

In [ ]:
# Compute a 126-day (roughly half-year) rolling correlation; the result is a
# `Series` indexed by date that shows how the SPY-TLT relationship evolves
# over time.
rolling_corr = pair["SPY"].rolling(126).corr(pair["TLT"])

# Where We Are Heading Next


These time-series tools will support later chapters on modeling, portfolio
construction, and reporting.


## Figure Generation (Optional)
Run the chapter's figure scripts under `code/figures/` to regenerate the PNG
files under `assets/figures/`.


In [ ]:
import runpy

scripts = [
    "../code/figures/ch09_prices_overview.py",
    "../code/figures/ch09_sma_crossover.py",
    "../code/figures/ch09_sma_equity_curve.py",
]

for script in scripts:
    try:
        runpy.run_path(script, run_name="__main__")
        print(f"OK: {script}")
    except ModuleNotFoundError as e:
        print(f"Skipping {script}: missing dependency ({e.name}).")
    except Exception as e:
        print(f"Failed {script}: {type(e).__name__}: {e}")


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
